# Preclass

AI for Materials Science — Hands-on session 8: Generative models for new molecules and crystal structures

A generative model learns from examples and then proposes new ones. In class you will train a model that proposes
molecules and use a pretrained model that proposes crystals, and then check whether the proposals make sense.
This notebook prepares the two ideas and the two data formats you need for that. It takes about 30–40 minutes,
runs on a CPU runtime, and nothing is submitted.

## What this notebook covers

- **0. Setup:** install two libraries and download the course data.
- **A. A GAN on MNIST:** a generator and a discriminator that train against each other.
- **B. A VAE on MNIST:** an encoder, a decoder and a latent space you can sample from.
- **C. Molecules as graphs:** from a SMILES string to the tensors a neural network reads, and back.
- **D. Crystals as structures:** a lattice, atoms at fractional coordinates, and distances across cell borders.

**On a shared computer:** use *File → Save a copy in Drive* to keep your work, and at the end use *Runtime → Disconnect and delete runtime*, then sign out of Google.

## 0. Setup

### 0-1. Install the libraries

Run this cell first. It installs RDKit, a chemistry toolkit for molecules, and pymatgen, a toolkit for crystal structures. Colab already includes PyTorch, NumPy, pandas and Matplotlib.

In [ ]:
!pip install -q rdkit==2026.9.1 pymatgen==2026.10.24

### 0-2. Download the course material

This cell copies the course repository into a folder called `MS49900-AI4M`. The notebook reads these files from its `Data/` folder:

- `qm9_subset_20k.csv`: 20,000 small organic molecules from QM9 (section C).
- `mp_LiPS_references.json.gz`: known Li–P–S crystals from the Materials Project (section D).

If git says the destination already exists, it leaves that folder unchanged and does not update the files. If a later cell reports a missing file, check the contents of `MS49900-AI4M/Data/`.

In [ ]:
!git clone https://github.com/kwongibaek/MS49900-AI4M.git

### 0-3. Import the libraries

The next cell imports the tools used throughout the notebook.

In [ ]:
import gzip, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torchvision import datasets
from rdkit import Chem
from rdkit.Chem import BondType, Draw
from pymatgen.core import Structure

### 0-4. Set the file paths

`DATA_DIR` points to the downloaded data. `Data folder exists: True` confirms that the folder is there.

In [ ]:
DATA_DIR = Path("MS49900-AI4M/Data")
print("Data folder exists:", DATA_DIR.exists())
torch.manual_seed(0)

## A. A GAN on MNIST

A generative adversarial network (GAN) has two networks. The **generator** turns 100 random numbers into an image.
The **discriminator** looks at an image and outputs a score: high for "real", low for "generated". They train in
turns: the discriminator learns to tell real from generated images, and the generator learns to fool it. When the
training works, the generated images look like the real ones.

### A-1. Load the images of the digit 2

MNIST has 60,000 training images of handwritten digits, 28 × 28 pixels each. We keep only the digit 2, as in last
year's class, so the generator has one shape to learn. Each image becomes a row of 784 numbers scaled to [−1, 1],
because the generator's last layer (`tanh`) outputs numbers in that range.

In [ ]:
mnist = datasets.MNIST("mnist", train=True, download=True)
x_two = mnist.data[mnist.targets == 2].float().div(255).mul(2).sub(1).reshape(-1, 784)
print("images of the digit 2:", x_two.shape)

def show_grid(images, title):
    fig, axes = plt.subplots(4, 4, figsize=(4.2, 4.2))
    for ax, img in zip(axes.flat, images):
        ax.imshow(img.reshape(28, 28), cmap="gray", vmin=-1, vmax=1); ax.axis("off")
    fig.suptitle(title); plt.show()

show_grid(x_two[:16], "Real images of the digit 2")

### A-2. The generator and the discriminator

The layer sizes are the same as in last year's model. `LeakyReLU` lets a small signal through for negative inputs,
and `Dropout` switches off 30 % of the discriminator's units at random during training, so it cannot memorise the
real images. The discriminator returns a raw score (a logit); the loss function turns it into a probability.

In [ ]:
LATENT = 100
generator = nn.Sequential(
    nn.Linear(LATENT, 256, bias=False), nn.BatchNorm1d(256), nn.LeakyReLU(0.2),
    nn.Linear(256, 512, bias=False), nn.BatchNorm1d(512), nn.LeakyReLU(0.2),
    nn.Linear(512, 784), nn.Tanh())
discriminator = nn.Sequential(
    nn.Linear(784, 512), nn.LeakyReLU(0.2), nn.Dropout(0.3),
    nn.Linear(512, 256), nn.LeakyReLU(0.2), nn.Dropout(0.3),
    nn.Linear(256, 1))
print(generator)

### A-3. Train the two networks in turns

Each iteration has two discriminator steps and one generator step (`D_STEPS = 2`).

- **Discriminator step:** real images get the target 0.9 instead of 1.0 ("label smoothing", which keeps the
  discriminator from becoming overconfident) and generated images get the target 0.
- **Generator step:** the generator is rewarded when the discriminator gives its images the target of a real image.

The losses do not fall steadily as in ordinary training: each network's loss goes up when the other one improves.
This takes about two minutes on a Colab CPU.

In [ ]:
BATCH, N_ITER, D_STEPS, SMOOTH = 128, 4000, 2, 0.9
bce = nn.BCEWithLogitsLoss()
opt_d = torch.optim.Adam(discriminator.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_g = torch.optim.Adam(generator.parameters(), lr=2e-4, betas=(0.5, 0.999))
real_target = torch.full((BATCH, 1), SMOOTH)
fake_target = torch.zeros(BATCH, 1)

for it in range(N_ITER):
    for _ in range(D_STEPS):
        real = x_two[torch.randint(len(x_two), (BATCH,))]
        fake = generator(torch.randn(BATCH, LATENT)).detach()
        d_loss = bce(discriminator(real), real_target) + bce(discriminator(fake), fake_target)
        opt_d.zero_grad(); d_loss.backward(); opt_d.step()
    g_loss = bce(discriminator(generator(torch.randn(BATCH, LATENT))), real_target)
    opt_g.zero_grad(); g_loss.backward(); opt_g.step()
    if it % 1000 == 0 or it == N_ITER - 1:
        print(f"iteration {it:4d} | discriminator {d_loss.item():.3f} | generator {g_loss.item():.3f}")

### A-4. Look at the generated digits

Each image below starts from different random numbers. If most of them look like a 2, the generator has learned
the shape. If all of them look almost the same, the generator has found one image that fools the discriminator
and stopped exploring; this is called *mode collapse*.

*Try it yourself:* change `x_two` in A-1 to another digit (for example `mnist.targets == 9`) and train again.

In [ ]:
generator.eval()
with torch.no_grad():
    samples = generator(torch.randn(16, LATENT))
show_grid(samples, "Generated from random numbers")

## B. A VAE on MNIST

A variational autoencoder (VAE) has an **encoder** and a **decoder**. The encoder does not map an image to a single
point; it outputs a mean `mu` and a log-variance `log_var` that describe a small cloud of points in a
**latent space**. We draw one point `z` from that cloud and the decoder turns it back into an image.

The loss has two parts:

- **reconstruction:** how well the decoded image matches the input;
- **KL divergence:** how far the encoder's clouds are from a standard normal distribution. This term keeps the latent
  space compact, so a random point drawn from a standard normal distribution decodes to a sensible image.

### B-1. All digits, scaled to [0, 1]

In [ ]:
x_all = mnist.data.float().div(255).reshape(-1, 784)
labels = mnist.targets
test = datasets.MNIST("mnist", train=False, download=True)
x_test, y_test = test.data.float().div(255).reshape(-1, 784), test.targets
print(x_all.shape, x_test.shape)

### B-2. Encoder, decoder and loss

The sizes are those of last year's model: 784 → 512 → 2 and back. A two-dimensional latent space is too small for
sharp images, but we can draw it. `z = mu + sigma * epsilon` is the *reparameterisation trick*: the random part
`epsilon` is separate, so the gradient can flow through `mu` and `sigma`.

In [ ]:
class VAE(nn.Module):
    def __init__(self, hidden=512, latent=2):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(784, hidden), nn.ReLU())
        self.mu, self.log_var = nn.Linear(hidden, latent), nn.Linear(hidden, latent)
        self.decoder = nn.Sequential(nn.Linear(latent, hidden), nn.ReLU(), nn.Linear(hidden, 784), nn.Sigmoid())

    def forward(self, x):
        h = self.encoder(x)
        mu, log_var = self.mu(h), self.log_var(h)
        z = mu + torch.exp(0.5 * log_var) * torch.randn_like(mu)
        return self.decoder(z), mu, log_var

def vae_loss(x, x_hat, mu, log_var):
    reconstruction = nn.functional.binary_cross_entropy(x_hat, x, reduction="sum") / len(x)
    kl = -0.5 * torch.sum(1 + log_var - mu**2 - log_var.exp()) / len(x)
    return reconstruction + kl, reconstruction, kl

vae = VAE()

### B-3. Train for 10 epochs

The total loss should fall quickly in the first epochs. This takes about one minute.

In [ ]:
opt = torch.optim.Adam(vae.parameters(), lr=1e-3)
loader = torch.utils.data.DataLoader(x_all, batch_size=128, shuffle=True)
for epoch in range(10):
    total = 0.0
    for x in loader:
        loss, rec, kl = vae_loss(x, *vae(x))
        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item() * len(x)
    print(f"epoch {epoch + 1:2d} | loss per image {total / len(x_all):.1f}")

### B-4. The latent space

Left: each test image is placed at its mean `mu` and coloured by its digit. Similar digits sit close together,
although the model never saw the labels. Right: we decode a grid of latent points. Moving across the grid changes
one digit smoothly into another; this is what "sampling from the latent space" means.

In [ ]:
vae.eval()
with torch.no_grad():
    mu_test = vae.mu(vae.encoder(x_test)).numpy()
    grid = torch.tensor([[a, b] for b in np.linspace(2, -2, 12) for a in np.linspace(-2, 2, 12)], dtype=torch.float32)
    decoded = vae.decoder(grid).reshape(12, 12, 28, 28).permute(0, 2, 1, 3).reshape(12 * 28, 12 * 28)
plt.figure(figsize=(5.4, 5))
sc = plt.scatter(mu_test[:, 0], mu_test[:, 1], c=y_test, cmap="tab10", s=3)
plt.colorbar(sc, ticks=range(10)); plt.xlabel("z[0]"); plt.ylabel("z[1]")
plt.title("Test images in the latent space"); plt.show()
plt.figure(figsize=(5, 5))
plt.imshow(decoded, cmap="gray"); plt.axis("off"); plt.title("Decoded grid of latent points"); plt.show()

## C. Molecules as graphs

A molecule can be written as a **SMILES** string, for example `CCO` for ethanol. For a neural network we turn it
into a graph: the heavy atoms are the nodes and the bonds are the edges (hydrogen atoms are implied).

In class we use **QM9**, a set of 134,000 small organic molecules with at most 9 heavy atoms (C, N, O, F) and
properties calculated with density functional theory, such as the HOMO–LUMO gap. The file here is a random sample
of 20,000 of them.

### C-1. Read a few molecules

In [ ]:
qm9 = pd.read_csv(DATA_DIR / "qm9_subset_20k.csv")
print(qm9.shape)
display(qm9.head())
Draw.MolsToGridImage([Chem.MolFromSmiles(s) for s in qm9.smiles[:8]], molsPerRow=4, subImgSize=(200, 150),
                     legends=[f"gap {g:.2f} eV" for g in qm9.gap_eV[:8]])

### C-2. From SMILES to tensors

Two arrays describe one molecule. These are the same functions the in-class model uses.

- **Adjacency tensor**, shape `(BOND_DIM, NUM_ATOMS, NUM_ATOMS)` = (5, 9, 9): one 9 × 9 table per bond type (single,
  double, triple, aromatic) plus one table for "no bond". Entry `[b, i, j]` is 1 if atoms i and j share a bond of
  type b.
- **Feature matrix**, shape `(NUM_ATOMS, ATOM_DIM)` = (9, 5): one row per atom slot, with a 1 in the column of its
  element (C, N, O, F) or in the last column if the slot is empty.

In [ ]:
ATOMS = ["C", "N", "O", "F"]                       ## heavy atoms in QM9 (hydrogens stay implicit)
BOND_TYPES = [BondType.SINGLE, BondType.DOUBLE, BondType.TRIPLE, BondType.AROMATIC]
NUM_ATOMS = 9                                      ## QM9 molecules have at most 9 heavy atoms
ATOM_DIM = len(ATOMS) + 1                          ## + one column for "no atom"
BOND_DIM = len(BOND_TYPES) + 1                     ## + one channel for "no bond"

def smiles_to_graph(smiles):
    """SMILES -> adjacency tensor (BOND_DIM, NUM_ATOMS, NUM_ATOMS) and feature matrix (NUM_ATOMS, ATOM_DIM)."""
    mol = Chem.MolFromSmiles(smiles)
    adjacency = np.zeros((BOND_DIM, NUM_ATOMS, NUM_ATOMS), "float32")
    features = np.zeros((NUM_ATOMS, ATOM_DIM), "float32")
    for atom in mol.GetAtoms():
        i = atom.GetIdx()
        features[i, ATOMS.index(atom.GetSymbol())] = 1
        for neighbor in atom.GetNeighbors():
            j = neighbor.GetIdx()
            b = BOND_TYPES.index(mol.GetBondBetweenAtoms(i, j).GetBondType())
            adjacency[b, [i, j], [j, i]] = 1
    adjacency[-1, adjacency.sum(axis=0) == 0] = 1   ## pairs without a bond use the "no bond" channel
    features[features.sum(axis=1) == 0, -1] = 1      ## empty rows use the "no atom" column
    return adjacency, features


adjacency, features = smiles_to_graph(qm9.smiles[3])
print("SMILES:", qm9.smiles[3])
print("adjacency", adjacency.shape, "| features", features.shape)
print("atoms in the slots:", [(ATOMS + ["-"])[k] for k in features.argmax(axis=1)])
print("single bonds (channel 0):\n", adjacency[0].astype(int))

### C-3. Back from tensors to a molecule

A generative model outputs such tensors, so we also need the way back. `graph_to_molecule` rebuilds the molecule and
returns `None` when the tensors break a chemical rule, for example a carbon atom with five bonds. Here we check the
round trip on 500 molecules. The graph has no stereochemistry (the `@` and `/` marks in a SMILES), so we compare the
SMILES without it. Some molecules still change: a single heavy atom (such as methane) has no bonds and is lost, and
an aromatic N–H (as in pyrrole) loses its hydrogen.

In [ ]:
def graph_to_molecule(adjacency, features):
    """Tensors -> RDKit molecule, or None if RDKit cannot sanitize it."""
    keep = np.where((features.argmax(axis=1) != ATOM_DIM - 1) & (adjacency[:-1].sum(axis=(0, 1)) != 0))[0]
    features, adjacency = features[keep], adjacency[:, keep][:, :, keep]
    mol = Chem.RWMol()
    for t in features.argmax(axis=1):
        mol.AddAtom(Chem.Atom(ATOMS[t]))
    for b, i, j in zip(*np.where(np.triu(adjacency) == 1)):
        if i == j or b == BOND_DIM - 1:
            continue
        mol.AddBond(int(i), int(j), BOND_TYPES[b])
    if Chem.SanitizeMol(mol, catchErrors=True) != Chem.SanitizeFlags.SANITIZE_NONE:
        return None
    return mol


def flat_smiles(mol):
    return Chem.MolToSmiles(mol, isomericSmiles=False)    ## canonical SMILES without stereochemistry

same = 0
for s in qm9.smiles[:500]:
    mol = graph_to_molecule(*smiles_to_graph(s))
    same += mol is not None and flat_smiles(mol) == flat_smiles(Chem.MolFromSmiles(s))
print(f"round trip unchanged for {same} of 500 molecules")

## D. Crystals as structures

A crystal repeats one unit cell in all directions. Three lattice vectors define the cell, and each atom has
**fractional coordinates**: its position as fractions of those vectors. pymatgen stores this as a `Structure`.
In class, MatterGen proposes such structures and you check them.

### D-1. Load Li2S from the Materials Project reference file

The file holds known Li–P–S crystals from the Materials Project. We take the most stable Li2S entry.

In [ ]:
refs = json.load(gzip.open(DATA_DIR / "mp_LiPS_references.json.gz", "rt"))
entry = min((e for e in refs if e["formula_pretty"] == "Li2S"), key=lambda e: e["energy_above_hull"])
li2s = Structure.from_dict(entry["structure"])
print(entry["material_id"], entry["formula_pretty"], "| energy above hull", entry["energy_above_hull"], "eV/atom")
print(li2s)

### D-2. Lattice, fractional and Cartesian coordinates

`frac_coords` are fractions of the lattice vectors; `coords` are positions in ångström. The density follows from the
atoms' masses and the cell volume.

In [ ]:
lat = li2s.lattice
print("a, b, c (Å):", np.round(lat.abc, 3), "| angles (°):", np.round(lat.angles, 1))
print(f"volume {lat.volume:.2f} Å³ | density {li2s.density:.3f} g/cm³ | {len(li2s)} atoms in the cell")
for site in li2s:
    print(site.specie, "fractional", np.round(site.frac_coords, 3), "| Cartesian (Å)", np.round(site.coords, 3))

### D-3. Distances across the cell border

Because the cell repeats, the nearest neighbour of an atom can sit in the next cell, and it can even be a copy of the
same atom. `get_neighbor_list(r)` lists every atom and periodic copy within r Å of each atom; only the atom itself is
left out. Each atom has a copy one lattice length away, so a radius just above the shortest of a, b and c always finds
the nearest neighbour. In class this is the first check of a generated crystal: two atoms closer than about 0.5 Å
cannot exist. The last lines print the start of the structure as a CIF file, the text format MatterGen writes.

In [ ]:
r = min(lat.abc) + 0.01
centres, neighbours, images, distances = li2s.get_neighbor_list(r)
k = distances.argmin()
print(f"shortest distance: {distances[k]:.3f} Å, from atom {centres[k]} ({li2s[centres[k]].specie}) to atom "
      f"{neighbours[k]} ({li2s[neighbours[k]].specie}) in the cell shifted by {images[k].astype(int)}")
print(li2s.to(fmt="cif")[:600])

### D-4. What comes next

In class you will (1) train a VAE whose encoder reads the tensors from C-2 and whose decoder writes them, and
(2) let MatterGen propose Li–P–S solid-electrolyte crystals and check them with the tools from D: the elements,
the shortest distance, the charge balance, and a comparison with known crystals such as this Li2S.

## Data sources

- **QM9:** R. Ramakrishnan, P. O. Dral, M. Rupp, O. A. von Lilienfeld, *Scientific Data* 1, 140022 (2014),
  https://doi.org/10.1038/sdata.2014.22. Data: figshare, https://doi.org/10.6084/m9.figshare.c.978904, licensed under
  CC0 1.0 (https://creativecommons.org/publicdomain/zero/1.0/). `Data/qm9_subset_20k.csv` is a random sample of
  20,000 molecules (uncharacterised and charged molecules removed); see `Data/qm9_SOURCE.txt`.
- **Materials Project:** `Data/mp_LiPS_references.json.gz`, Li–P–S(–Cl, –Br) phases with energy above hull
  ≤ 0.1 eV/atom, accessed 2026-11-04, licensed under CC BY 4.0 (https://creativecommons.org/licenses/by/4.0/).
  Cite: A. Jain et al., *APL Materials* 1, 011002 (2013), https://doi.org/10.1063/1.4812323.
- **MNIST:** Y. LeCun, C. Cortes, C. J. C. Burges, downloaded at run time with torchvision; not redistributed with
  the course material.
- Sections A and B follow last year's course notebooks, which were prepared with reference to the machine learning
  course materials of Seungchul Lee (KAIST) and Sergei Kalinin (University of Tennessee). The graph functions in C
  follow the Keras example "Drug Molecule Generation with VAE" by Victor Basu (Apache License 2.0).